# Face Hair - uruchomienie po instalacji
Ten notebook pomija ponowna instalacje, gdy srodowisko jest gotowe. Do wygenerowania wlosow potrzebne sa oficjalne wagi DiffLocks.
Zarejestruj sie na https://difflocks.is.tue.mpg.de/ . Uruchom wszystkie komorki i wpisz login/haslo DiffLocks w Colabie, albo nacisnij Enter, by wgrac pobrany ZIP. Nie podawaj danych w czacie.


In [ ]:
from pathlib import Path
import os, sys
if Path("/content/facehair_env/bin/python").exists() and Path("/content/difflocks_launcher.py").exists():
    print("Srodowisko juz zainstalowane. Pomijam ponowna instalacje.")
else:
    import os, sys, subprocess, urllib.request
    from pathlib import Path

    def run(*args, env=None):
        subprocess.run(args, check=True, env=env)

    print("Face Hair - naprawiony start DiffLocks Studio")
    run("nvidia-smi")
    run(sys.executable, "-m", "pip", "install", "-q", "uv")
    run("uv", "python", "install", "3.12")
    run("uv", "venv", "--python", "3.12", "--seed", "/content/facehair_env")

    env = os.environ.copy()
    env["PATH"] = "/content/facehair_env/bin:" + env.get("PATH", "")
    env["PYTHONNOUSERSITE"] = "1"
    env.pop("PYTHONPATH", None)
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            env["HF_TOKEN"] = token
    except Exception:
        pass

    python = "/content/facehair_env/bin/python"
    run(python, "-m", "pip", "install", "-q", "ipywidgets", "ipython", "requests", "huggingface_hub", env=env)

    url = "https://raw.githubusercontent.com/arqdariogomez/DiffLocks-Studio/3db8c8f/launchers/Difflocks_colab_github_minimal.txt"
    source = urllib.request.urlopen(url, timeout=60).read().decode("utf-8")
    source = source.replace("https://shi-labs.com/natten/wheels/", "https://whl.natten.org/old")
    start = source.index("def render_debug_html(logs_list):")
    end = source.index("\n" + chr(39) * 3 + "\n", start)
    source = source[:start] + source[start:end].replace(chr(39) * 3, chr(34) * 3) + source[end:]
    import ast
    ast.parse(source)
    print("Poprawka skladni DiffLocks: OK", flush=True)
    source = source.replace("pip install --no-cache-dir torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0", "pip install --no-cache-dir torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0")
    source = source.replace('HF_ASSETS_REPO = "arqdariogomez/difflocks-assets-hybrid"', 'HF_ASSETS_REPO = "arqdariogomez/difflocks-assets-hybrid"')
    file = Path("/content/Face_DiffLocks_Launcher.py")
    source = source.replace('        render_status(96, "🚀 Starting DiffLocks Studio...")', '        render_status(100, "Srodowisko gotowe. Przejdz do kolejnej komorki.", "success")\n        return')
    file.write_text(source)
    print("Przygotowuje srodowisko. To moze chwile potrwac.")
    run(python, "-u", str(file), env=env)

    

In [ ]:
from pathlib import Path
from getpass import getpass
from zipfile import ZipFile
import requests, shutil
root = Path("/content/DiffLocks-Studio")
ckpt = root / "checkpoints"
ckpt.mkdir(parents=True, exist_ok=True)
models = list(ckpt.rglob("*.pth"))
codec = list(ckpt.rglob("strand_codec.pt"))
if models and codec:
    print("Wagi AI juz sa na miejscu. Pomijam pobieranie.")
else:
    print("Otworz https://difflocks.is.tue.mpg.de/ i zarejestruj sie, jesli nie masz konta.")
    print("Podaj login i haslo tylko w Colabie. Nie wysylaj ich do rozmowy.")
    print("Dane zostana przeslane jedynie na oficjalny serwer download.is.tue.mpg.de.")
    username = input("Login DiffLocks (Enter = wgraj pobrany ZIP): ").strip()
    zip_path = Path("/content/difflocks_checkpoints.zip")
    if username:
        password = getpass("Haslo DiffLocks: ")
        url = "https://download.is.tue.mpg.de/download.php?domain=difflocks&sfile=difflocks_checkpoints.zip"
        response = requests.post(url, data={"username": username, "password": password}, stream=True, timeout=(30,120))
        password = None
        response.raise_for_status()
        with zip_path.open("wb") as target:
            for chunk in response.iter_content(1024*1024):
                if chunk:target.write(chunk)
        print("Pobrano", round(zip_path.stat().st_size/1024**2,1), "MB")
    else:
        from google.colab import files
        uploaded=files.upload()
        if not uploaded:
            raise RuntimeError("Nie wybrano pliku.")
        path=Path("/content") / next(iter(uploaded))
        if path!=zip_path: shutil.move(str(path),zip_path)
        del uploaded
    if not ZipFile(zip_path).testzip() is None:
        raise RuntimeError("Archiwum wag jest uszkodzone.")
    with ZipFile(zip_path) as z:
        for info in z.infolist():
            path=Path(info.filename)
            if path.is_absolute() or ".." in path.parts:
                raise RuntimeError("Nieprawidlowa sciezka w archiwum.")
        z.extractall(root)
    models=list(ckpt.rglob("*.pth"))
    codec=list(ckpt.rglob("strand_codec.pt"))
    if not models or not codec:
        for item in root.rglob("strand_codec.pt"):
            if ckpt not in item.parents:
                destination=ckpt/"strand_vae"/item.name
                destination.parent.mkdir(parents=True,exist_ok=True)
                shutil.copyfile(item,destination)
                break
        for item in root.rglob("*.pth"):
            if ckpt not in item.parents and "scalp" in item.name.lower():
                destination=ckpt/"difflocks_diffusion"/item.name
                destination.parent.mkdir(parents=True,exist_ok=True)
                shutil.copyfile(item,destination)
        models=list(ckpt.rglob("*.pth"))
        codec=list(ckpt.rglob("strand_codec.pt"))
    if not models or not codec:
        raise RuntimeError("W archiwum nie ma wymaganych wag (.pth i strand_codec.pt).")
    print("Wagi AI znalezione:", len(models), "model(i) oraz",len(codec),"kodek(i).")


In [ ]:
from pathlib import Path
import os, subprocess, re
from IPython.display import display, HTML
python=Path("/content/facehair_env/bin/python")
launcher=Path("/content/difflocks_launcher.py")
if not python.exists() or not launcher.exists():
    raise RuntimeError("Brakuje srodowiska. Uruchom pierwsza komorke.")
if not list(Path("/content/DiffLocks-Studio/checkpoints").rglob("strand_codec.pt")):
    raise RuntimeError("Brakuje wag AI. Uruchom druga komorke.")
env=os.environ.copy()
env["PATH"]="/content/facehair_env/bin:"+env.get("PATH","")
env["PYTHONNOUSERSITE"]="1"
env.pop("PYTHONPATH",None)
print("Uruchamiam DiffLocks Studio. Otworz link gradio.live, gdy sie pojawi.")
proc=subprocess.Popen([str(python), "-u", str(launcher)], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line,end="",flush=True)
    match=re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live",line)
    if match:
        display(HTML('<p><a href="'+match.group(0)+'" target="_blank">OTWORZ GENERATOR WLOSOW</a></p>'))
code=proc.wait()
if code:
    raise RuntimeError(f"DiffLocks zakonczyl sie bledem {code}, szczegoly sa powyzej.")
